# 01. CAD 데이터 탐색

DXF 파일에서 추출한 Feature 데이터 탐색 및 분석

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.extractor import FeatureExtractor

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. 데이터 로드

In [ ]:
# CSV 파일 목록
csv_files = list(config.PROCESSED_DIR.glob('*_full.csv'))
print(f'CSV 파일 수: {len(csv_files)}')
for f in csv_files:
    print(f'  - {f.name}')

In [ ]:
# 샘플 데이터 로드
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(10)

## 2. 기본 통계

In [ ]:
# 엔티티 타입 분포
print('=== Entity Type Distribution ===')
print(df['entity_type'].value_counts())

In [ ]:
# 레이어 분포
print('=== Layer Distribution ===')
layer_counts = df['layer'].value_counts()
print(layer_counts.head(15))

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entity Type
df['entity_type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Entity Type Distribution')
axes[0].set_xlabel('Entity Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Layer (Top 10)
layer_counts.head(10).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 10 Layers')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. 레이어명 분석

In [ ]:
# 모든 고유 레이어명
all_layers = df['layer'].unique().tolist()
print(f'고유 레이어 수: {len(all_layers)}')
print('\n레이어 목록:')
for layer in sorted(all_layers):
    print(f'  - {layer}')

In [ ]:
# 규칙 기반 분류 테스트
from src.classifier import RuleBasedClassifier

classifier = RuleBasedClassifier()
predictions = classifier.predict(all_layers)

layer_classification = pd.DataFrame({
    'layer': all_layers,
    'predicted_class': predictions
})
print('=== Layer Classification ===')
print(layer_classification)

In [ ]:
# 분류 결과 분포
print('\n=== Classification Distribution ===')
print(layer_classification['predicted_class'].value_counts())

## 4. 좌표 분석 (LINE 엔티티)

In [ ]:
# LINE 엔티티만 필터
lines_df = df[df['entity_type'] == 'LINE'].copy()
print(f'LINE 엔티티 수: {len(lines_df)}')
lines_df.head()

In [ ]:
# 좌표 범위
print('=== Coordinate Range ===')
print(f"X: {lines_df['start_x'].min():.2f} ~ {lines_df['end_x'].max():.2f}")
print(f"Y: {lines_df['start_y'].min():.2f} ~ {lines_df['end_y'].max():.2f}")

In [ ]:
# LINE 시각화 (레이어별 색상)
fig, ax = plt.subplots(figsize=(12, 10))

# 레이어별로 다른 색상
layers = lines_df['layer'].unique()
colors = plt.cm.tab20(range(len(layers)))
layer_colors = dict(zip(layers, colors))

for _, row in lines_df.iterrows():
    color = layer_colors[row['layer']]
    ax.plot(
        [row['start_x'], row['end_x']],
        [row['start_y'], row['end_y']],
        color=color, linewidth=0.5, alpha=0.7
    )

ax.set_aspect('equal')
ax.set_title('LINE Entities Visualization')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.show()

## 5. 다음 단계

- `02_feature_engineering.ipynb`: 레이어명 특성 추출
- `03_model_training.ipynb`: ML 모델 학습